In [2]:
from datasets import load_dataset

# loading review data
dataset = load_dataset('json', data_files='Subscription_Boxes.jsonl.gz')

df = dataset['train'].to_pandas()

In [3]:
import pandas as pd

In [4]:
df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1.0,USELESS,Absolutely useless nonsense and a complete was...,[],B07G584SHG,B09WC47S3V,AEMJ2EG5ODOCYUTI54NBXZHDJGSQ,1602133857705,2,True
1,2.0,Manufactured where?,"With a couple of the items, I wasn't quite sur...",[],B07QL1JRCN,B07QL1JRCN,AEEJBFZKUBEEMBZUZJV4UHFVEEBQ,1609110735433,20,True
2,1.0,Little bang for your buck.,Two SMALL stuffed animals and 2 little bags of...,[],B07RBYJN37,B08N5QKX1Y,AGSVZNZBTSGQBKZDZTQYEZHGDPCQ,1609937315319,4,True
3,5.0,New favorite box,"Although I don’t remember signing up for this,...",[],B07KM6T8GV,B07KM6T8GV,AFDERNB6BIR3U2DOR3S2KX7KJJXQ,1616156351887,1,True
4,5.0,Coctique,I loved every thing and could use it all. Thin...,[],B07NVL6TJG,B07NVKNVNM,AE6P2YJ6FKX332MD56GPJFSHXNJQ,1559533206066,0,True


In [5]:
import numpy as np

# converting ratings into a float ranging between 0-1 (will be treated as the target)

df['rating'] = np.divide(df['rating'], 5.0)
df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,0.2,USELESS,Absolutely useless nonsense and a complete was...,[],B07G584SHG,B09WC47S3V,AEMJ2EG5ODOCYUTI54NBXZHDJGSQ,1602133857705,2,True
1,0.4,Manufactured where?,"With a couple of the items, I wasn't quite sur...",[],B07QL1JRCN,B07QL1JRCN,AEEJBFZKUBEEMBZUZJV4UHFVEEBQ,1609110735433,20,True
2,0.2,Little bang for your buck.,Two SMALL stuffed animals and 2 little bags of...,[],B07RBYJN37,B08N5QKX1Y,AGSVZNZBTSGQBKZDZTQYEZHGDPCQ,1609937315319,4,True
3,1.0,New favorite box,"Although I don’t remember signing up for this,...",[],B07KM6T8GV,B07KM6T8GV,AFDERNB6BIR3U2DOR3S2KX7KJJXQ,1616156351887,1,True
4,1.0,Coctique,I loved every thing and could use it all. Thin...,[],B07NVL6TJG,B07NVKNVNM,AE6P2YJ6FKX332MD56GPJFSHXNJQ,1559533206066,0,True


In [6]:
x = df['text']
y = df['rating']

In [7]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

In [8]:

# tokenizing inputs
tokenizer = RobertaTokenizer.from_pretrained('FacebookAI/roberta-base', truncation=True, padding=True)

def tokenize(text):
        return tokenizer(text, truncation=True, padding=True)

x_tokenized = x.map(lambda text: tokenize(text))

In [9]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x_tokenized, y, test_size=.3, random_state=42)



In [10]:
train = pd.DataFrame([x_train, y_train])
train = train.transpose()

test = pd.DataFrame([x_test, y_test])
test = test.transpose()

train
test

,text,rating
2678,"[input_ids, attention_mask]",0.6
14247,"[input_ids, attention_mask]",0.4
476,"[input_ids, attention_mask]",0.6
15857,"[input_ids, attention_mask]",0.4
5563,"[input_ids, attention_mask]",1.0
...,...,...
7195,"[input_ids, attention_mask]",0.2
10866,"[input_ids, attention_mask]",0.2
4238,"[input_ids, attention_mask]",0.8
7220,"[input_ids, attention_mask]",0.8


In [ ]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

train['text'].apply(lambda text: group_texts)


,text,rating
13163,"[input_ids, attention_mask]",0.2
1472,"[input_ids, attention_mask]",1.0
6839,"[input_ids, attention_mask]",1.0
15658,"[input_ids, attention_mask]",1.0
7517,"[input_ids, attention_mask]",1.0
...,...,...
13418,"[input_ids, attention_mask]",1.0
5390,"[input_ids, attention_mask]",1.0
860,"[input_ids, attention_mask]",0.2
15795,"[input_ids, attention_mask]",0.2


In [19]:
# input_dict = {'input_ids': [k['input_ids'] for k in x_train],
#             'attention_mask': [k['attention_mask'] for k in x_train]}
train

,text,rating
13163,"[input_ids, attention_mask]",0.2
1472,"[input_ids, attention_mask]",1.0
6839,"[input_ids, attention_mask]",1.0
15658,"[input_ids, attention_mask]",1.0
7517,"[input_ids, attention_mask]",1.0
...,...,...
13418,"[input_ids, attention_mask]",1.0
5390,"[input_ids, attention_mask]",1.0
860,"[input_ids, attention_mask]",0.2
15795,"[input_ids, attention_mask]",0.2


In [14]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

data_collater = DataCollatorWithPadding(tokenizer=tokenizer)

In [15]:
model = RobertaForSequenceClassification.from_pretrained('FacebookAI/roberta-base')

training_args = TrainingArguments(
    output_dir="./roberta-sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",   # use "f1" if classes are imbalanced
    greater_is_better=True,
    warmup_ratio=0.1,
    logging_steps=50,
    save_total_limit=2,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=test,
    data_collator=data_collater,
    processing_class=tokenizer,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


KeyError: 7875